# M7 · Capstone — your reproducible optimization PR

**Learning objective:** assemble a genuine experimental report from **your own** W7900 records — measured baseline, measured candidate, declared protocol, quality scope, limitations, and a reproduction command — and draft the optimization PR. If your physical evidence is smoke/classroom-only, the report is labelled a **PILOT**, never a proven general improvement.

Reference lesson: [M07 challenge](../course/M07_student_challenge/CHALLENGES.md) · [中文](./08_capstone_and_pr.zh-CN.ipynb)

In [ ]:
from pathlib import Path
import json
import statistics
import sys
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "lab/reference_runs.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the course repository root (see notebooks/README.md).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
COURSE_MODE = "REFERENCE"  # <- edit once: REFERENCE | GPU_CLASSROOM | GPU_RESEARCH
RUN_OWN_GPU_ARMS = True   # <- False in GPU modes: reuse the latest physical run (shown run_id)
from lab import notebook_runtime as rt
from lab import notebook_support as ns
session = rt.begin(COURSE_MODE, notebook="08")

def _stamp(kind: str, value: str) -> None:
    """Print a visible origin tag so no number is mistaken for another kind."""
    print(f"[{kind}] {value}")

MEASURED_TODAY = "measured on THIS GPU in THIS session (physical)"
HISTORICAL   = "frozen reference evidence from the committed campaign"
ESTIMATE     = "analytical estimate or simulation - NOT a measurement"

## 1 · Declare your experiment
Fill in your identity and pick the eligible module whose records you will report (a module with a genuine physical measurement from this course: M1, M3, M4, M5, M6).

In [ ]:
STUDENT_ID = "your-name-here"          # <- edit
STUDENT_NOTE = "what you claim and why it matters"
ELIGIBLE_MODULE = "M4"                 # M1|M3|M4|M5|M6 (module with YOUR physical run)
CANDIDATE_EXPERIMENT = {"M1": ("B0", "B0"), "M3": ("B0", "L1"),
                        "M4": ("B0", "F1"), "M5": ("A", "C"),
                        "M6": ("B0", "BEST")}.get(ELIGIBLE_MODULE, ("B0", "BEST"))
BASE_EXPERIMENT, CAND_EXPERIMENT = CANDIDATE_EXPERIMENT
print(f"capstone: {STUDENT_ID} reports {ELIGIBLE_MODULE}: "
      f"{BASE_EXPERIMENT} vs {CAND_EXPERIMENT}")

## 2 · Collect the evidence (own run first, campaign fallback)
The collector prefers **your own** physical runs from this session, then your earlier runs under `runs/`, and only then the shared qualification-campaign records (explicitly labelled as shared). Reference-only capstones are labelled as such.

In [ ]:
def collect_evidence(exp):
    own = [e for e in session.executions if e.get("experiment") == exp]
    if own:
        b = rt.load_run(own[-1]["run_id"])
        return b, "own physical run (this session)"
    rid = session.latest_campaign_run(exp)
    if rid and session.mode != "REFERENCE":
        return rt.load_run(rid), f"shared campaign record {rid}"
    rid = rt.list_runs(exp)
    if rid:
        return rt.load_run(rid[-1]), f"existing run {rid[-1]}"
    return None, "no physical record"

base_b, base_src = collect_evidence(BASE_EXPERIMENT)
cand_b, cand_src = collect_evidence(CAND_EXPERIMENT)
for nm, b, src in ((BASE_EXPERIMENT, base_b, base_src),
                   (CAND_EXPERIMENT, cand_b, cand_src)):
    print(f"{nm:5s} {src}" + (f" run_id={b.run_id}" if b else ""))

## 3 · Compute the claim and its correct scope label

In [ ]:
import subprocess

def git_head():
    r = subprocess.run(["git", "-C", str(ROOT), "rev-parse", "HEAD"],
                       capture_output=True, text=True, timeout=10)
    return r.stdout.strip() or "unknown"

report = {
    "student_id": STUDENT_ID, "module": ELIGIBLE_MODULE,
    "claim": STUDENT_NOTE,
    "git_head": git_head(),
    "gpu": session.admission.detail if session.admission else None,
    "baseline": {"experiment": BASE_EXPERIMENT, "source": base_src,
                 "stats": base_b.stats()} if base_b else None,
    "candidate": {"experiment": CAND_EXPERIMENT, "source": cand_src,
                  "stats": cand_b.stats()} if cand_b else None,
}
if base_b and cand_b:
    ab = rt.ab_compare(base_b, cand_b)
    report["comparison"] = {k: ab[k] for k in
                            ("classification", "speedup_median",
                             "vram_delta_mb", "paired_per_prompt")}
    if ab["classification"] == "FORMAL_PROTOCOL_MEASURED":
        report["evidence_class"] = "PHYSICAL_FORMAL"
    elif "own physical run" in base_src:
        report["evidence_class"] = "PHYSICAL_PILOT"
    else:
        report["evidence_class"] = "SHARED_CAMPAIGN_PHYSICAL"
else:
    report["evidence_class"] = "REFERENCE_ONLY"
print("evidence_class:", report["evidence_class"])
print(json.dumps(report.get("comparison", {}), indent=1)[:600])

## 4 · Write the report (append-only, under `runs/`)
Never into frozen `evidence/`. The file name is unique; re-running creates a new record instead of overwriting.

In [ ]:
report["limitations"] = [
    "speedup is a ratio of the two declared populations only",
    "single-host, single-GPU (Radeon Cloud W7900 instance)",
    "quality verdict stays PENDING_HUMAN_REVIEW unless you completed the M6 rubric",
]
report["reproduce"] = (
    f"python -m lab run --experiment {CAND_EXPERIMENT} --date <your-tag>")
from datetime import datetime
_cap_ts = datetime.now().strftime("%H%M%S")
out = ns.student_record(report, f"capstone-{_cap_ts}", confirmed=True)
print("capstone report:", out)

## 5 · Your PR draft
Copy the template below into your PR description; every bracket must be replaced by YOUR values, and the title must carry the evidence class from section 3.

In [ ]:
pr_draft = f"""# optimization: [{report['evidence_class']}] {ELIGIBLE_MODULE} {BASE_EXPERIMENT} vs {CAND_EXPERIMENT}

## Hypothesis
[what you predicted and why]

## Protocol
- GPU: {(report['gpu'] or {}).get('device_name', 'n/a')} @ ROCm {(report['gpu'] or {}).get('hip', '')}
- populations: baseline {(report['baseline'] or {}).get('stats', {}).get('population', 'n/a')} | candidate {(report['candidate'] or {}).get('stats', {}).get('population', 'n/a')}
- run records: {(report['baseline'] or {}).get('stats', {}).get('run_id', '')} , {(report['candidate'] or {}).get('stats', {}).get('run_id', '')}

## Results
- speedup median: {report.get('comparison', {}).get('speedup_median')}
- VRAM delta: {report.get('comparison', {}).get('vram_delta_mb')} MB

## Quality
- [rubric verdict or PENDING_HUMAN_REVIEW]

## Limitations
{chr(10).join('- ' + l for l in report['limitations'])}

## Reproduce
{report['reproduce']}
"""
print(pr_draft)

In [ ]:
summary = {
    "mode": session.mode,
    "notebook": session.notebook,
    "executions": [e.get("run_id") or e.get("kind") for e in session.executions],
    "consumed": [c.get("run_id") for c in session.consumed],
    "note": "append-only session record; physical evidence lives under runs/",
}
session.finish(summary)

## Acceptance criteria

- [ ] You can state which numbers are **measured on this GPU today**, which are
      **historical reference values**, and which are **estimates/simulations**.
- [ ] Every physical run you produced has a visible `run_id` under `runs/`.
- [ ] You did not present a smoke/classroom population as a formal result.
- [ ] You can explain one limitation of each measurement you reported.